# Supervised Learning

Choose **one regression task and one classification task**. For each, examine the data, justify preprocessing, compare models, and explain the results.
Assessment considers performance on unseen test data, training error, and the quality of the report.

## 1. Prediction tasks

Each dataset contains **5,000 labeled observations**. Feature counts exclude identifiers and targets.
See the corresponding `data_dictionary.md` for individual column definitions.

### Robot joint torque · Real regression

**Dataset:** `sarcos_regression`.

**Target:** `torque_1`, the torque at the robot's first joint, on the original numerical scale.

**Inputs (21):** Seven joint positions, seven velocities, and seven accelerations.

**Remark:** Feature scales differ; consider their effect on regularized models.  
Negative and extreme values may describe valid operating conditions.  
Physical units are incompletely documented; report errors on the source scale.

Source: [SARCOS](https://gaussianprocess.org/gpml/data/).

### Building cooling load · Synthetic regression

**Dataset:** `cooling_regression`.

**Target:** `cooling_load_kw`, the total thermal cooling load of four building zones, in kW.

**Inputs (20):** Four zones, each with area, temperature difference, heat-transfer coefficient, solar irradiance, and internal heat gain.

**Remark:** Temperature differences are outdoor minus indoor; some observations are missing.  
Missing temperatures are not zeros, while zero solar irradiance is valid.  
The simplified simulation describes thermal cooling load, not electrical power consumption.

### Credit card default · Real classification

**Dataset:** `credit_classification`.

**Target:** `default_next_month`: 1 means recorded payment default in the following month; 0 means no default.
Default means failing to meet repayment obligations. Predict the **probability of class 1**.

**Inputs (20):** Credit limit, age, and six months each of repayment status, bill amounts, and payment amounts.

**Remark:** Repayment status codes are not necessarily equally spaced; justify their encoding.  
Negative codes and bill amounts can be valid; check before transforming them.  
Examine class balance; these records describe customers in 2005.

Source: [UCI credit card default](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients).

### Order execution risk · Synthetic classification

**Dataset:** `execution_classification`.

**Target:** `high_slippage`: 1 means adverse execution slippage exceeds 5 basis points (0.05%); 0 means otherwise.
Predict the **probability of class 1**.

**Inputs (20):** Bid/ask prices and quantities at four levels, plus order size, direction, execution delay, and recent volatility.

**Remark:** Buys consume asks; sells consume bids, and `side=-1` is valid.  
Slippage is measured from the best executable quote at submission, not the mid-price.  
Future market changes are random; the simulation is not calibrated to a real market.

## 2. Required experiments

Complete the following for **both selected tasks**, except where stated otherwise.

| Experiment | What to do |
|---|---|
| Data diagnosis | Examine feature meanings, ranges, missingness, skewness, and unusual values. For classification, also examine class proportions. Justify numerical or categorical treatment. |
| Preprocessing comparison | Compare two versions that differ in one justified preprocessing choice. Keep the learner, folds, and tuning budget the same; report the difference for each fold. |
| Baseline | Use Linear Regression or Ridge for regression, and Logistic Regression for classification. Explain preprocessing and parameter choices. |
| Bagging | Fit the baseline with the same feature representation on bootstrap samples and average predictions or class-1 probabilities. Record the number of models and other settings. |
| Improvement (optional) | Add justified features or use another model. Explain the change and compare it fairly with the baseline. |
| Learning curve | For **at least one task**, compare training and validation error at **three or more training sizes**. Subsample only the training portion of each fold; keep validation rows fixed. |

Possible preprocessing comparisons include scaling for robot torque, imputation for cooling load, categorical encoding of repayment status, and relative prices for execution risk.
Keep other necessary preprocessing in both comparison conditions.

Explain why each change helped, had little effect, or reduced performance. Bagging need not improve a model that underfits. Describe any changes to regularization or other baseline settings separately.
For the learning curve, discuss what the results suggest about model complexity, sample size, and noise; preserve class proportions when subsampling classification data.

## 3. Setup and data

Place this notebook, `train.py`, `test.py`, and the existing `datasets/` folder in the same working directory.
This layout works in local Jupyter or Colab; in Colab, upload the files and extract the data archive first.

- Edit `make_model` in **`train.py`** to define preprocessing, models, and hyperparameters.
- Edit `predict` in **`test.py`** only if the default prediction method does not support your model. Leave metric calculations unchanged.
- Use this notebook for experiments and analysis. Do not copy the two scripts into notebook cells.

Run the example for one dataset at a time. Change `DATASET` and repeat for your other selected task.

In [ ]:
# In Colab, upload train.py, test.py, and datasets.zip.
# In local Jupyter, place these files beside this notebook.
try:
    from google.colab import files
except ImportError:
    print("Local Jupyter: using the files in the working directory.")
else:
    uploaded = files.upload()


In [ ]:
# Extract the supplied dataset archive if the folder is not already present.
from pathlib import Path
from zipfile import ZipFile

if not Path("datasets").is_dir():
    with ZipFile("datasets.zip") as archive:
        archive.extractall(".")
print("Dataset folder:", Path("datasets").resolve())


In [ ]:

%pip install -q "numpy>=1.26" "pandas>=2.1" "scikit-learn>=1.4,<2" "matplotlib>=3.7"
from pathlib import Path
import importlib
import json
import pandas as pd
import train as training
import test as testing

DATA_ROOT = Path("datasets")
DATASET = "cooling_regression"  # Choose a directory listed in Section 1.
SEED = 2026
# Setup completed.

In [ ]:
data_dir = DATA_ROOT / DATASET
metadata = json.loads((data_dir / "metadata.json").read_text())
data = pd.read_csv(data_dir / "train.csv", dtype={"sample_id": str})
X = data[metadata["feature_names"]]
y = data[metadata["target"]]

print(f"{len(data)} observations, {X.shape[1]} features; target: {metadata['target']}")
display(data.head())
display(X.describe().T.assign(missing_rate=X.isna().mean()))
if metadata["task"] == "binary_classification":
    display(y.value_counts(normalize=True).sort_index().rename("class_fraction"))
# Data loaded

## 4. Training and validation

Use the supplied **five-fold cross-validation** assignments in `cv_folds.csv`.
The 5,000 observations are divided into five groups of 1,000. Train on four groups and validate on the remaining group, rotating as follows:

| Run | Training folds | Validation fold |
|---|---|---|
| 1 | 1, 2, 3, 4 | 0 |
| 2 | 0, 2, 3, 4 | 1 |
| 3 | 0, 1, 3, 4 | 2 |
| 4 | 0, 1, 2, 4 | 3 |
| 5 | 0, 1, 2, 3 | 4 |

Each run uses **4,000 training rows and 1,000 validation rows**. Fit a fresh model and fit all learned preprocessing on the training rows only.
Keep the same folds when comparing methods. Neither `sample_id` nor `fold` is a model input.
The separate held-out test set is reserved for final evaluation.

The next cell checks the supplied folds and prepares a directory for experiment scores. `training.validate()` aligns folds by `sample_id` and fits a fresh model for each split.

In [ ]:
folds = pd.read_csv(data_dir / "cv_folds.csv", dtype={"sample_id": str})
assert data["sample_id"].is_unique and folds["sample_id"].is_unique
assert set(data["sample_id"]) == set(folds["sample_id"])
fold_id = data["sample_id"].map(folds.set_index("sample_id")["fold"])
assert set(fold_id) == set(range(5))

run_dir = Path("runs") / DATASET
run_dir.mkdir(parents=True, exist_ok=True)


`train(data_path)` returns a fitted model bundle, including preprocessing.
`test(data_path, model)` returns a report whose `metrics` field contains the scores.
The example below runs the supplied baseline. After implementing another method in `train.py`, change `METHOD` and rerun this cell.

| Task | Main metric | Also report | Additional metrics computed automatically |
|---|---|---|---|
| Regression | RMSE | MAE | MSE, R² |
| Classification | Log loss | ROC-AUC | Accuracy, F1 |

Classification models output class-1 probabilities.
Accuracy and F1 use threshold 0.5; log loss clips probabilities to `[1e-15, 1 - 1e-15]`.

In [ ]:
importlib.reload(training)  # Pick up edits to train.py and test.py.
importlib.reload(testing)
METHOD = "baseline"  # Change after implementing another method.

results, summary = training.validate(data_dir / "train.csv", method=METHOD, seed=SEED)
results.to_csv(run_dir / f"{METHOD}_scores.csv", index=False)
display(results)
display(summary)


Add the preprocessing comparison, other models, and learning curve below, with plots and explanations.
The fold scores for each method are saved under `runs/`; retain them when comparing methods or changing parameters.

In [ ]:
# Add code for the remaining experiments here.

**Analysis**

Describe the evidence for your preprocessing choices, compare training and validation results, and interpret the learning curve.
State the settings used in each experiment and discuss the limitations of your conclusions.

## 5. Train the selected model

After model selection, fit the chosen configuration on all 5,000 development observations.
Keep the returned model bundle in memory; it contains the model and its fitted preprocessing.
Evaluating these same rows gives training error, not test error.


In [ ]:
FINAL_METHOD = "baseline"  # Replace with the method selected using validation.
importlib.reload(training)
model = training.train(data_dir / "train.csv", method=FINAL_METHOD, seed=SEED)
print(f"Trained {DATASET} ({FINAL_METHOD}) on all {len(data):,} development rows.")


For a separate labeled test CSV, pass the model bundle in the same Python session:

```python
report = testing.test("path/to/test.csv", model)
print(report["metrics"])
```

No preprocessing or model fitting takes place during testing. Without true labels, error cannot be calculated.
After a runtime restart, rerun training to recreate the model bundle.


## 6. Report and submission

Submit the completed notebook or report, `train.py`, `test.py`, any additional modules, and instructions for reproducing the results.
Identify both selected datasets and document dependencies, random seeds, preprocessing, features, and hyperparameters.
Include data diagnosis, preprocessing and model comparisons, a learning curve, and an interpretation of the results.
Final test administration will be specified separately.

Use only the supplied data for training. Do not retrieve public-source labels to match records or obtain held-out answers.
Cite external resources used.

**AI policy.** AI tools are permitted, but disclose the tools and how you used them—for example, model implementation or deployment,
hyperparameter search, data analysis, or report drafting and editing. Identify the affected code, experiments, or report sections.
If you did not use AI, state this. You remain responsible for the correctness of the work and for explaining it.

**Readability affects the grade.** Organize the report logically, use consistent notation, label figures and tables, and explain what the results show.

**AI Use Disclosure:** _State the tools used and their contributions, or state that no AI tools were used._